# Step 3 Behavioral Three-Zone Workbench

This notebook is a separate Step 3 workbench for the GigScore project.
It wraps the current behavioral pipeline in `STEP-3/test_on_master_gig.py` so you can inspect features, train the model, review zone metrics, and preview Agent 4 payloads without duplicating the production logic.


In [ ]:
from pathlib import Path
import importlib.util
import json

import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "STEP-3").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

STEP3_SCRIPT = PROJECT_ROOT / "STEP-3" / "test_on_master_gig.py"
assert STEP3_SCRIPT.exists(), f"Missing Step 3 script: {STEP3_SCRIPT}"

spec = importlib.util.spec_from_file_location("step3_master_eval", STEP3_SCRIPT)
step3 = importlib.util.module_from_spec(spec)
spec.loader.exec_module(step3)

PROJECT_ROOT, STEP3_SCRIPT


In [ ]:
raw = pd.read_csv(step3.MASTER_CSV)
y = raw["TARGET"].astype(int)
red_flags = step3.build_red_flag_frame(raw)
X = step3.build_behavioral_features(raw, red_flags)

print(f"Rows: {len(raw):,}")
print(f"Columns in master dataset: {raw.shape[1]}")
print(f"Behavioral feature count: {X.shape[1]}")
pd.Series(X.columns, name="feature_name").to_frame().head(15)


In [ ]:
results = step3.train_and_evaluate(X, y, raw, red_flags, n_folds=step3.CV_FOLDS)
report_text = step3.generate_report(results)
print(report_text)


In [ ]:
diagnostic = pd.DataFrame([step3.serialize_metric(results["diagnostic_metrics"])])
zone_summary = pd.DataFrame(results["zone_metrics"]["zones"]).T

display(diagnostic)
display(zone_summary)


In [ ]:
agent4_payload = step3.build_agent4_payload(
    results["raw_test"],
    results["zone_frame"],
    results["portfolio_safe_rate"],
)

print(agent4_payload.keys())
print(agent4_payload["applicants"][0])


## Persisting Production Artifacts

If you want the exact production files regenerated from the notebook session, run the script directly:

```python
!python STEP-3/test_on_master_gig.py
```

That writes the metrics JSON, predictions CSV, Agent 4 payload, report, and model bundle into the same project locations used by the script.
